# Actor-Critic: learn online with one-step TD errors

Vanilla Actor-Critic combines a stochastic **actor** $\pi_\theta(a\mid s)$ with a **critic** $V_\phi(s)$. After every transition, the critic constructs the one-step TD target

$$y_t=r_{t+1}+\gamma(1-d_t)V_\phi(s_{t+1}),$$

where $d_t$ is one only for a true terminal state. The one-step TD error

$$\delta_t=y_t-V_\phi(s_t)$$

trains the critic and acts as the actor's advantage estimate. This is the canonical online formulation: collect one transition, update both networks once, and continue with the new policy.


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

ENV_ID = "CartPole-v1"
TOTAL_TIMESTEPS = 20_000
ACTOR_LEARNING_RATE = 3e-4
CRITIC_LEARNING_RATE = 1e-3
GAMMA = 0.99

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = gym.make(ENV_ID)
observation_dim = int(np.prod(env.observation_space.shape))
action_dim = env.action_space.n
print(f"Observation size: {observation_dim}; actions: {action_dim}; device: {device}")

## 1. Build separate actor and critic networks

The actor outputs one logit per discrete action. A categorical distribution converts the logits into action probabilities. The critic outputs one scalar estimate $V_\phi(s)$. Small, separate networks and optimizers keep the two learning problems visible.


In [ ]:
def make_network(output_dim):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(observation_dim, 64),
        nn.Tanh(),
        nn.Linear(64, output_dim),
    ).to(device)


actor = make_network(output_dim=action_dim)
critic = make_network(output_dim=1)
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=ACTOR_LEARNING_RATE)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=CRITIC_LEARNING_RATE)


def action_distribution(observations):
    observations = torch.as_tensor(observations, dtype=torch.float32, device=device)
    return torch.distributions.Categorical(logits=actor(observations))


def select_action(observation, deterministic=False):
    with torch.no_grad():
        distribution = action_distribution(np.atleast_2d(observation))
        action = (
            distribution.probs.argmax(dim=-1)
            if deterministic
            else distribution.sample()
        )
    return int(action.item())

## 2. Update from one transition

For a sampled transition $(s_t,a_t,r_{t+1},s_{t+1})$, the critic minimizes the squared TD error:

$$L_{\text{critic}}=\frac{1}{2}\left(V_\phi(s_t)-y_t\right)^2.$$

The actor treats the detached TD error as a sampled advantage:

$$L_{\text{actor}}=-\operatorname{stopgrad}(\delta_t)\log \pi_\theta(a_t\mid s_t).$$

A positive TD error makes the sampled action more probable; a negative error makes it less probable. Detaching $\delta_t$ prevents the actor loss from changing the critic. No rollout, return normalization, entropy bonus, or multi-step estimator is used.

Gymnasium distinguishes termination from truncation. A true termination has no future state value, so its bootstrap is zero. A time-limit truncation still bootstraps from $V_\phi(s_{t+1})$.


In [ ]:
def update(observation, action, reward, next_observation, terminated):
    observation = torch.as_tensor(
        np.atleast_2d(observation), dtype=torch.float32, device=device
    )
    next_observation = torch.as_tensor(
        np.atleast_2d(next_observation), dtype=torch.float32, device=device
    )

    value = critic(observation).squeeze()
    with torch.no_grad():
        next_value = critic(next_observation).squeeze()
        td_target = reward + GAMMA * (1.0 - float(terminated)) * next_value
    td_error = (td_target - value).detach()

    distribution = action_distribution(observation)
    action = torch.as_tensor(action, dtype=torch.int64, device=device)
    actor_loss = -distribution.log_prob(action).squeeze() * td_error
    critic_loss = 0.5 * (value - td_target).square()

    actor_optimizer.zero_grad()
    actor_loss.backward()
    actor_optimizer.step()

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    return {
        "actor_loss": actor_loss.item(),
        "critic_loss": critic_loss.item(),
        "td_error": td_error.item(),
    }

## 3. Learn online

Sample one action, take one environment step, and immediately update the actor and critic from that transition. Because the networks change after every step, each transition is used exactly once by the policy that generated it. CartPole is retained because this online formulation learned consistently in the installed environment without custom reward shaping.


In [ ]:
def train(total_timesteps):
    episode_returns, metrics = [], []
    episode_return = 0.0
    observation, _ = env.reset()

    for step in range(1, total_timesteps + 1):
        action = select_action(observation)
        next_observation, reward, terminated, truncated, _ = env.step(action)
        metrics.append(
            update(observation, action, reward, next_observation, terminated)
        )
        episode_return += reward

        if terminated or truncated:
            episode_returns.append(episode_return)
            episode_return = 0.0
            observation, _ = env.reset()
        else:
            observation = next_observation

        print(
            f"\rStep {step}/{total_timesteps} | "
            f"Episodes: {len(episode_returns)} | "
            f"Episode return: {episode_return:.1f}",
            end="",
        )

    env.close()
    return episode_returns, metrics


episode_returns, metrics = train(TOTAL_TIMESTEPS)
print(f"\nTrained for {len(episode_returns)} completed episodes.")

## 4. Inspect learning

Episode return measures performance. Losses and TD errors need not decrease monotonically because the changing policy changes the data distribution. Smoothed curves make their scale and trend easier to inspect.


In [ ]:
def moving_average(values, window):
    values = np.asarray(values)
    window = min(window, len(values))
    return np.convolve(values, np.ones(window) / window, mode="valid")


returns = np.asarray(episode_returns)
return_window = min(20, len(returns))
update_window = min(200, len(metrics))
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
axes[0].plot(returns, alpha=0.3, label="episode return")
axes[0].plot(
    np.arange(return_window - 1, len(returns)),
    moving_average(returns, return_window),
    label=f"{return_window}-episode mean",
)
axes[0].set(title=f"Actor-Critic on {ENV_ID}", xlabel="Episode", ylabel="Return")
axes[0].legend()
for axis, key, title in zip(
    axes[1:],
    ["td_error", "critic_loss", "actor_loss"],
    ["TD error", "Critic loss", "Actor loss"],
    strict=True,
):
    values = [item[key] for item in metrics]
    axis.plot(
        np.arange(update_window - 1, len(values)),
        moving_average(values, update_window),
    )
    axis.set(title=title, xlabel="Environment step")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Evaluate the modal policy

Training samples actions. Evaluation selects the highest-probability action and uses a separate environment so it cannot disturb training state.


In [ ]:
env = gym.make(ENV_ID, render_mode="human")
episode_returns = []

for episode in range(5):
    observation, _ = env.reset()
    episode_return = 0.0
    for step in range(1000):
        action = select_action(observation, deterministic=True)
        observation, reward, terminated, truncated, _ = env.step(action)
        episode_return += reward
        print(
            f"Episode {episode + 1}: step={step + 1}, " f"return={episode_return:.1f}",
            end="\r",
        )
        if terminated or truncated:
            break
    episode_returns.append(episode_return)
    print()

env.close()
print(f"Mean return: {np.mean(episode_returns):.1f} +/- {np.std(episode_returns):.1f}")